# Hands-On Lab

### Train and Deploy an XGBoost Model on AWS SageMaker

**1. What you will do**

Upload a small dataset to Amazon S3.
Open a notebook in SageMaker Studio.
Train an XGBoost model with SageMaker's built-in algorithm.
Deploy it to a real-time endpoint.
Send 5 test customers to the endpoint and compare with the known answers.
Delete the endpoint and confirm nothing is running.
Check your cost.

**2. Files you need**

+ rain.csv: 3,200 rows, target in column 1, no header → upload to S3
+ validation.csv: 800 rows, same format → upload to S3
+ test_rows.csv: 5 customers, no target column → testing the endpoint
+ bank_sample_with_header.csv: all 4,000 rows with column names → open it to
understand the data

The 14 columns, in order:



| # | Column Name | Type / Values | Description |
|---|---|---|---|
| 1 | **y** | 0 = did not, 1 = subscribed | Target (has the client subscribed to a term deposit?) |
| 2 | **age** | Numeric | Age in years |
| 3 | **job** | Categorical: 0 = admin, 1 = technician, 2 = services, 3 = management, 4 = retired, 5 = student | Type of job |
| 4 | **marital** | Categorical: 0 = single, 1 = married, 2 = divorced | Marital status |
| 5 | **education** | Categorical: 0 = primary, 1 = secondary, 2 = tertiary, 3 = unknown | Level of education |
| 6 | **default** | Binary: 0 = no, 1 = has credit in default | Has credit in default? |
| 7 | **balance** | Numeric | Average yearly balance in euros |
| 8 | **housing** | Binary: 0 = no, 1 = has a housing loan | Has housing loan? |
| 9 | **loan** | Binary: 0 = no, 1 = has a personal loan | Has personal loan? |
| 10 | **contact** | Categorical: 0 = cellular, 1 = telephone | Contact communication type |
| 11 | **duration** | Numeric (seconds) | Last contact duration in seconds |
| 12 | **campaign** | Numeric | Number of contacts performed during this campaign for this client |
| 13 | **previous** | Numeric | Number of contacts performed before this campaign for this client |
| 14 | **poutcome** | Categorical: 0 = unknown, 1 = previous failure, 2 = previous success | Outcome of the previous marketing campaign |

### **3. Part 0: Safety setup**

***Step 0.1 Sign in and choose ONE region***

+ Sign in to the AWS Console.
+ At the top right, pick a region (for example US East (N. Virginia)) and use the
same region for the whole lab.

***Step 0.2 Create a budget alert***

+ In the search bar, type Budgets and open AWS Budgets.
+ Click Create budget, choose the simple template Monthly cost budget (names can
vary slightly).
+ Set the amount to 1.00 (USD) and enter your email for alerts.
+ Create the budget.

***Step 0.3 Write down your start time***

### **4. Part 1: Put the data in S3**

***Step 1.1 Create a bucket***

+ Search S3, open it, click Create bucket.
+ Bucket name: must be unique across all of AWS, lowercase, for example
yourname-lecture8-lab-2026.
+ Region: the same region as Step 0.1.
+ Leave everything else default. Click Create bucket.

***Step 1.2 Upload the data***
+ Open your bucket, click Create folder, name it processed, click Create folder.
+ Open the processed folder, click Upload, Add files, choose train.csv and
validation.csv, click Upload.

### **5. Part 2: Open a notebook**

+ The Studio screens change often, so follow the intent of each step.

***Step 2.1 Open SageMaker Studio***
+ Search SageMaker AI and open it.
+ Find Studio (or "Get started") and open it.
+ First time only: create a domain with the quick/default setup. This takes a few
minutes. Wait until it is ready.
+ Click Open Studio.

***Step 2.2 Start a notebook environment***
+ In Studio, open JupyterLab (or "Notebook"). If asked to create a "space" or choose
an instance, choose ml.t3.medium (this is the free-tier notebook size).

+ Start it, then open it when it is ready.
+ Create a new Python 3 notebook.
Step 2.3 Upload nothing, just copy code
+ You will paste the code cells below into the notebook, one cell at a time, and run
each with Shift+Enter.

### **6. Part 3: Train the model**
Cell 1: check the SDK version

In [18]:
!pip install sagemaker==2.112.0
!pip install boto3==1.26.16
!pip install awscli
!aws configure

AWS Access Key ID [****************None]: None
AWS Secret Access Key [****************None]: None
Default region name [None]: None
Default output format [None]: None


In [19]:
import sagemaker
print(sagemaker.__version__)

2.112.0


+ You should see: a version number starting with 2 (for example 2.2xx.x).
+ If it starts with 3: this lab uses the version-2 API. Run the next lines, then restart
the kernel (Kernel menu, Restart) and run Cell 1 again:

```
!pip install -q "sagemaker<3"
```

Cell 2: set up your names
+ Replace the bucket name with yours.

In [20]:
import sagemaker, boto3
from sagemaker import image_uris
from sagemaker.estimator import Estimator
from sagemaker.inputs import TrainingInput

# Giả lập thông tin cấu hình AWS để không bị lỗi trên Colab
# For real project
"""
session = sagemaker.Session()
role = sagemaker.get_execution_role()
region = session.boto_region_name
"""

region = "us-east-1"
role = "arn:aws:iam::123456789012:role/service-role/AmazonSageMaker-ExecutionRole-20260101T000001"
bucket = "lecture8-bucketNN"

print("Region:", region)
print("Role:", role)
print("Bucket:", bucket)

Region: us-east-1
Role: arn:aws:iam::123456789012:role/service-role/AmazonSageMaker-ExecutionRole-20260101T000001
Bucket: lecture8-bucket


Cell 3: define the training job (free-tier size, with a time limit)

In [21]:
# Sử dụng phiên bản xgboost được hỗ trợ bởi phiên bản SDK hiện tại (ví dụ: 1.5-1)
image = image_uris.retrieve("xgboost", region=region, version="1.5-1")

# Khởi tạo sagemaker session giả lập để tránh lỗi NameError
try:
    session_to_use = session
except NameError:
    session_to_use = sagemaker.LocalSession()  # Sử dụng LocalSession giả lập trên môi trường không có AWS

estimator = Estimator(
    image_uri=image,
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",        # free-tier instance type
    max_run=1800,                        # hard stop after 30 minutes
    output_path=f"s3://{bucket}/output",
    sagemaker_session=session_to_use,
)

estimator.set_hyperparameters(
    objective="binary:logistic",
    eval_metric="auc",                   # report AUC (higher is better)
    num_round=100,
    max_depth=5,
    eta=0.2,
)
print("Estimator ready")

Estimator ready


Cell 4: start training

In [22]:
"""
train_input = TrainingInput(f"s3://{bucket}/processed/train.csv",
content_type="text/csv")
val_input = TrainingInput(f"s3://{bucket}/processed/validation.csv",
content_type="text/csv")
estimator.fit({"train": train_input, "validation": val_input})
"""

'\ntrain_input = TrainingInput(f"s3://{bucket}/processed/train.csv",\ncontent_type="text/csv")\nval_input = TrainingInput(f"s3://{bucket}/processed/validation.csv",\ncontent_type="text/csv")\nestimator.fit({"train": train_input, "validation": val_input})\n'

In [31]:
from google.colab import files
uploaded = files.upload()

Saving bank_sample_with_header.csv to bank_sample_with_header.csv
Saving test_rows.csv to test_rows.csv
Saving train.csv to train.csv
Saving validation.csv to validation.csv


In [32]:
import os
# Thiết lập biến môi trường AWS mặc định để LocalSession không bị lỗi 'Unsupported region: None'
os.environ["AWS_DEFAULT_REGION"] = "us-east-1"

train_input = TrainingInput("file://train.csv", content_type="text/csv")
val_input = TrainingInput("file://validation.csv", content_type="text/csv")

# Do không kết nối trực tiếp với AWS thực tế, fit() trong Local Mode vẫn có thể yêu cầu Docker cài đặt cục bộ.
# Nếu bạn chỉ muốn chạy kiểm tra cú pháp, chúng ta sẽ để lệnh này làm ghi chú hoặc chạy thử nghiệm:
try:
    estimator.fit({"train": train_input, "validation": val_input})
except Exception as e:
    print("Lưu ý: Không thể thực thi huấn luyện SageMaker thực tế trên Colab không có cấu hình AWS/Docker.")
    print("Chi tiết lỗi:", e)

Lưu ý: Không thể thực thi huấn luyện SageMaker thực tế trên Colab không có cấu hình AWS/Docker.
Chi tiết lỗi: Unsupported region: None. You may need to upgrade your SDK version (pip install -U sagemaker) for newer regions. Supported region(s): af-south-1, ap-east-1, ap-northeast-1, ap-northeast-2, ap-northeast-3, ap-south-1, ap-southeast-1, ap-southeast-2, ca-central-1, cn-north-1, cn-northwest-1, eu-central-1, eu-north-1, eu-south-1, eu-west-1, eu-west-2, eu-west-3, me-south-1, sa-east-1, us-east-1, us-east-2, us-gov-west-1, us-west-1, us-west-2.


In [33]:
import pandas as pd
import xgboost as xgb

# Đọc dữ liệu từ file csv cục bộ
train_df = pd.read_csv('train.csv', header=None)
val_df = pd.read_csv('validation.csv', header=None)

# Chia đặc trưng (X) và nhãn (y). Cột đầu tiên (chỉ số 0) là nhãn
X_train, y_train = train_df.iloc[:, 1:], train_df.iloc[:, 0]
X_val, y_val = val_df.iloc[:, 1:], val_df.iloc[:, 0]

# Huấn luyện mô hình XGBoost cục bộ
local_model = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='auc',
    n_estimators=100,
    max_depth=5,
    learning_rate=0.2
)

local_model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

print("Đã huấn luyện xong mô hình XGBoost cục bộ!")

Đã huấn luyện xong mô hình XGBoost cục bộ!


In [34]:
class MockPredictor:
    def __init__(self, model):
        self.model = model

    def predict(self, data):
        # Giả lập hàm dự đoán nhận chuỗi CSV hoặc DataFrame
        import io
        if isinstance(data, str):
            df = pd.read_csv(io.StringIO(data), header=None)
        elif isinstance(data, bytes):
            df = pd.read_csv(io.StringIO(data.decode('utf-8')), header=None)
        else:
            df = pd.DataFrame(data)

        preds = self.model.predict_proba(df)[:, 1]
        return ",".join(map(str, preds)).encode('utf-8')

# Ghi đè phương thức deploy của estimator để trả về một MockPredictor
def mock_deploy(*args, **kwargs):
    print("Giả lập: Deploying endpoint thành công (Local Mode)")
    return MockPredictor(local_model)

estimator.deploy = mock_deploy

## **7. Part 4: Deploy and test**

Cell 5: deploy the endpoint (note the time!)

In [35]:
import time
from datetime import datetime

endpoint_name = "lecture8-lab-endpoint-" + str(int(time.time()))
print("Deploy started at:", datetime.now().strftime("%H:%M:%S"))

predictor = estimator.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.xlarge",        # free-tier instance type
    endpoint_name=endpoint_name,
)
print("Endpoint name:", endpoint_name)
print("Deploy finished at:", datetime.now().strftime("%H:%M:%S"))

Deploy started at: 05:48:25
Giả lập: Deploying endpoint thành công (Local Mode)
Endpoint name: lecture8-lab-endpoint-1794462505
Deploy finished at: 05:48:25


Cell 6: send the 5 test customers

In [42]:
import pandas as pd
import numpy as np

# Đọc 5 dòng test khách hàng từ file đã tải lên
test_file = 'test_rows.csv'
test_df = pd.read_csv(test_file, header=None)

print("Dữ liệu 5 khách hàng kiểm thử:")
display(test_df)

# Đặt tên cột từ 1 đến 13 để khớp hoàn hảo với tên đặc trưng lúc huấn luyện mô hình
test_df.columns = list(range(1, 14))

# Gửi dữ liệu qua predictor giả lập để lấy kết quả
response = predictor.predict(test_df)
predictions = response.decode('utf-8').split(',')

print("\nKết quả dự đoán xác suất đăng ký (Probability):")
for i, pred in enumerate(predictions):
    print(f"Khách hàng {i+1}: {float(pred):.4f}")

Dữ liệu 5 khách hàng kiểm thử:


,0,1,2,3,4,5,6,7,8,9,10,11,12
0,48,2,0,1,0,1229.45,1,0,0,5,6,0,0
1,47,0,0,1,0,1471.91,0,0,1,6,9,1,2
2,20,4,0,2,0,-495.96,0,0,1,11,2,0,0
3,35,2,0,1,0,-162.46,0,0,0,1371,2,0,0
4,37,0,1,2,0,555.57,1,0,1,1060,1,0,0



Kết quả dự đoán xác suất đăng ký (Probability):
Khách hàng 1: 0.0005
Khách hàng 2: 0.0062
Khách hàng 3: 0.0179
Khách hàng 4: 0.9878
Khách hàng 5: 0.9540
